In [2]:
"""
Sample Script for Downloading Report Content from SEC EDGAR
"""
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
import re


def get_latest_8k(ticker):
    headers = {'User-Agent': 'given.family@magnumwm.com'}

    # Get CIK from ticker
    ticker_cik = requests.get(
        "https://www.sec.gov/files/company_tickers.json",
        headers=headers
    ).json()
    cik = str([v['cik_str'] for k, v in ticker_cik.items() if v['ticker'] == ticker.upper()][0])

    # Get recent filings
    submissions = requests.get(
        f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json",
        headers=headers
    ).json()

    # Find latest 8-K
    recent = submissions['filings']['recent']
    df = pd.DataFrame(recent)
    latest_8k = df[df['form'] == '8-K'].iloc[0]

    accession = latest_8k['accessionNumber'].replace('-', '')
    primary_doc = latest_8k['primaryDocument']

    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{primary_doc}"
    return url


def download_8k_text(filing_url):
    headers = {'User-Agent': 'given.family@magnumwm.com'}

    response = requests.get(filing_url, headers=headers)
    response.raise_for_status()

    # removes HTML tables, scripts, styles
    soup = BeautifulSoup(response.text, 'html.parser')

    # Kill scripts, styles, and navigation junk
    for script in soup(["script", "style", "header", "footer", "nav"]):
        script.decompose()

    text = soup.get_text(separator='\n')

    # Clean up whitespace
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = '\n'.join(chunk for chunk in chunks if chunk)

    return text
    
# ==================== TEST ====================
if __name__ == "__main__":

    # Usage
    url = get_latest_8k("AAPL")
    print(url)
    clean_text = download_8k_text(url)
    print(clean_text)

https://www.sec.gov/Archives/edgar/data/320193/000114036126006577/ef20060722_8k.htm
true
true
true
true
true
true
NASDAQ
NASDAQ
NASDAQ
NASDAQ
NASDAQ
NASDAQ
NASDAQ
false
0000320193
0000320193
2026-02-24
2026-02-24
0000320193
aapl:Zero500NotesDue2031Member
2026-02-24
2026-02-24
0000320193
aapl:One375NotesDue2029Member
2026-02-24
2026-02-24
0000320193
us-gaap:CommonStockMember
2026-02-24
2026-02-24
0000320193
aapl:Three600NotesDue2042Member
2026-02-24
2026-02-24
0000320193
aapl:Three050NotesDue2029Member
2026-02-24
2026-02-24
0000320193
aapl:Two000NotesDue2027Member
2026-02-24
2026-02-24
0000320193
aapl:One625NotesDue2026Member
2026-02-24
2026-02-24
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM
8-K
CURRENT REPORT
Pursuant to Section 13 OR 15(d) of The Securities Exchange Act of 1934
February 24, 2026
Date of Report (Date of earliest event reported)
Apple Inc.
(Exact name of Registrant as specified in its charter)
California
(State or other jurisdiction
of in

In [ ]:
import pandas as pd
pd.read_excel("us_symbol_list.xlsx")

In [2]:
import pandas as pd
import requests

# 1) 读取 SEC 映射（download.py 里用的 company_tickers.json）
sec_raw = requests.get("https://www.sec.gov/files/company_tickers.json").json()
sec_df = pd.DataFrame.from_dict(sec_raw, orient="index")
sec_df = sec_df.rename(columns={"cik_str": "cik", "ticker": "ticker", "title": "title"})
sec_df["ticker"] = sec_df["ticker"].str.upper().str.strip()
sec_df["cik"] = sec_df["cik"].astype(str).str.zfill(10)

display(sec_df.head(10))
print("SEC mapping columns:", sec_df.columns.tolist())
print("SEC mapping rows:", len(sec_df))

# 2) 读取你的 Excel
excel_path = "us_symbol_list.xlsx"
symbols_df = pd.read_excel(excel_path)
print("Excel columns:", symbols_df.columns.tolist())
display(symbols_df.head(10))

# 3) 自动选择 symbol 列（常见列名）
candidates = ["symbol", "ticker", "Symbol", "Ticker", "SYMBOL", "TICKER"]
symbol_col = next((c for c in candidates if c in symbols_df.columns), symbols_df.columns[0])
print("Using symbol column:", symbol_col)

symbols_df["symbol_norm"] = symbols_df[symbol_col].astype(str).str.upper().str.strip()

# 4) 映射并检查一致性
merged = symbols_df.merge(
    sec_df[["ticker", "cik", "title"]],
    left_on="symbol_norm",
    right_on="ticker",
    how="left",
)

missing = merged[merged["cik"].isna()]
print("Total symbols:", len(symbols_df))
print("Matched symbols:", merged["cik"].notna().sum())
print("Missing symbols:", len(missing))

display(missing.head(20))

# 5) 看看映射样例
display(merged[merged["cik"].notna()].head(20))


JSONDecodeError: Expecting value: line 1 column 1 (char 0)